# Word Embeddings

**The problem with fixed encoders:**

- **Words are unrelated.** *good* is as far from *great* as from *table*.
- **Texts are just word counts.** Only which words appear, and how often.
- **Words don't interact.** *not* in *not good* doesn't change *good*.

**Word embeddings:** a **word embedding** maps each word to a vector of $D$ real numbers, usually **pretrained** on lots of text.

**From words to a text:** the encoder turns a text into **one** vector in two steps:

1. **Embed:** look up the vector of each token.
2. **Pool:** combine them into one vector $\mathbf h$, for example by **sum** or **mean**.

Pooling ignores word order: *dog bites man* and *man bites dog* get the same $\mathbf h$.

## Vector Semantics and Trainable Embedding Models

**Vector semantics:** represent each word's meaning as a vector. Most methods use the **distributional hypothesis**: words used in similar contexts have similar meanings.

**The model:** the parameters are the word vectors themselves, one per word plus one for `<unk>` (unknown words), initialised at random.

**Training data:** word pairs $(t, c)$ built from raw text: **1** if they co-occur, **0** for random pairs. No human labels needed, so data is plentiful.

**The classifier:** for a pair with vectors $\mathbf u$ and $\mathbf v$:

1. **Score:** $s = \mathbf u^\top \mathbf v$.
2. **Probability:** $p(1 | t, c) = \sigma(s)$.
3. **Loss:** $-\log p(y | t, c)$.
4. **Update:** gradient descent changes only $\mathbf u$ and $\mathbf v$.

The trained vectors are our **word embeddings**: words in similar contexts get similar vectors. This is how the skip-gram version of **word2vec** works: it learns vectors by predicting each word's neighbours.

### Questions

**Question 1.** We train on the pair (good, day) with label 1. The pmf for (good, day) changes, and so does the pmf for (day, good). Why?

<details>
<summary><b>Answer 1</b></summary>

Both pairs use the same two vectors, and $\mathbf u^\top \mathbf v = \mathbf v^\top \mathbf u$, so their pmfs are always equal.

</details>

**Question 2.** Other pmfs changed too. Which ones, and why?

<details>
<summary><b>Answer 2</b></summary>

Every pair with *good* or *day*, like (good, bad), since those two vectors moved. Pairs without them, like (another, regular), stay the same.

</details>

**Question 3.** Training on a single positive pair is not a full training strategy. Sketch a more complete one.

<details>
<summary><b>Answer 3</b></summary>

1. Make all positive pairs from the corpus.
2. For each positive pair, make a few negative pairs by picking random context words.
3. Shuffle the pairs and split them into mini-batches.
4. For each batch, average the loss over its pairs and update the vectors.
5. Repeat for several passes (epochs) over the data, until the loss stops improving.

</details>

## Pre-trained Word Embeddings

Training good embeddings needs huge amounts of text and compute. So we use **pre-trained** vectors that others have already trained.

### Vocabulary and Embedding Matrix

Pre-trained embeddings come in two parts:

- **Vocabulary:** gives each word a unique id: $0, 1, 2, \ldots, V-1$.
- **Embedding matrix** $\mathbf E$ of shape $V \times D$: row $i$ is the vector of word $i$.

Two special tokens are added at the start:

- `<pad>`: all zeros. It fills up short texts so a batch of texts has equal length.
- `<unk>`: the average of all vectors. It stands for any unknown word.

### Looking Up a Word

1. **Lowercase** the word.
2. **Find its id** in the vocabulary. Unknown words get the id of `<unk>`.
3. **Take that row** of $\mathbf E$.

**Example.** *City* → *city* → id 3 → $(0.9, 0.3, -0.5)$. *camera-ready* is not in the vocabulary → id 1 → the `<unk>` vector.

A single vector is just a list of numbers and hard to read on its own. The meaning shows in how vectors relate to each other.

## Structure in Word Embeddings

Words with something in common end up close together in the embedding space. **Sentiment** is one example.

NLTK's **opinion lexicon** lists English words as positive or negative:

| positive | negative |
|---|---|
| *good, excellent, happy* | *bad, awful, sad* |

Positive and negative words often end up in different parts of the space, so their vectors alone can tell them apart.

### How to See It

- **Plot:** reduce the vectors to 2 dimensions (for example with PCA) and plot them. Positive and negative words tend to form separate groups.
- **Cluster:** a clustering method like [k-means](https://scikit-learn.org/stable/modules/clustering.html#k-means) with 2 clusters roughly recovers the two labels.

**Limits:**

- No groups in a 2D plot doesn't mean sentiment is missing: squeezing many dimensions into 2 loses information.
- *good* and *bad* appear in very similar contexts (*a ___ day*), so their vectors can be close despite opposite sentiment.

### Questions

**Question 1.** Do you think an embedding space for English contains information about parts of speech (noun, verb, adjective, adverb)? Justify your answer with how embeddings are trained.

<details>
<summary><b>Answer 1</b></summary>

Yes. Embeddings learn from contexts, and each part of speech has typical contexts: nouns follow *the*, verbs follow *will*, adjectives come before nouns. So words of the same type get similar vectors.

</details>

**Question 2.** Could your answer change with the **language** of the training corpus? Justify your answer.

<details>
<summary><b>Answer 2</b></summary>

Yes. In languages with free word order, position says less about word type. In languages with many word forms (like Finnish), each form is rarer, so its vector is learned from less data.

</details>

**Question 3.** Can you think of a different definition of co-occurrence that would change what embeddings capture? You may assume extra linguistic information about the corpus is available.

<details>
<summary><b>Answer 3</b></summary>

Use **syntax** instead of distance: two words co-occur if they are linked in the sentence's parse, for example a verb and its object. Then *dog* and *cat* get close because they play the same roles, not because they appear near each other. The window size matters too: a small window captures more about word type, a large one more about topic.

</details>

## Similarity and Analogies

Embeddings let us measure how **similar** two words are, and even solve **analogies**.

### Cosine Similarity

**Cosine similarity** measures the angle between two vectors:

$$\cos(\mathbf u, \mathbf v) = \frac{\mathbf u^\top \mathbf v}{\|\mathbf u\|\,\|\mathbf v\|}$$

It goes from $-1$ (opposite directions) through $0$ (right angle) to $1$ (same direction), and ignores vector length.

**Example.** With made-up vectors *cat* $= (2, 1)$, *dog* $= (2, 2)$ and *car* $= (-1, 2)$:

- $\cos(\textit{cat}, \textit{dog}) = \frac{2 \cdot 2 + 1 \cdot 2}{\sqrt 5 \cdot \sqrt 8} = \frac{6}{6.32} \approx 0.95$: almost the same direction, so very similar.
- $\cos(\textit{cat}, \textit{car}) = \frac{2 \cdot (-1) + 1 \cdot 2}{\sqrt 5 \cdot \sqrt 5} = 0$: a right angle, so unrelated.

### Nearest Neighbours

The **top-k** words for a word $w$ are the $k$ words whose vectors have the highest cosine with the vector of $w$.

**Example.** The nearest neighbours of *city* are likely words like *town*, *downtown* and *capital*.

### Analogies

Adding and subtracting word vectors can solve analogies like *man → king, woman → queen*:

$$\mathbf u = \text{embed}(\textit{king}) - \text{embed}(\textit{man}) + \text{embed}(\textit{woman})$$

The word whose vector is closest to $\mathbf u$ is often *queen*.

### Bias

Embeddings learn from text written by people, and that text contains stereotypes and harmful views. The embeddings learn these too. For example, analogies with jobs and gender often give stereotyped answers.

### Questions

**Question 1.** Look at your results for cosine similarity, nearest neighbours and analogies. What part of how embeddings are trained explains what you saw?

<details>
<summary><b>Answer 1</b></summary>

Embeddings learn from co-occurrence, so "similar" means *used in similar contexts*, not *same meaning*. That's why related words (*city*, *place*) and even opposites (*good*, *bad*) end up close. Each word has one vector, so words with several meanings get a mix. And analogies copy patterns in the training text, including stereotypes.

</details>

**Question 2.** If the embeddings match human ideas of word similarity, why is **cosine** a useful way to compare two vectors, rather than another formula?

<details>
<summary><b>Answer 2</b></summary>

Training uses the dot product, so words in similar contexts get vectors pointing in similar **directions**. The dot product also grows with vector **length**, which mostly reflects how frequent a word is, not its meaning. Cosine keeps only the direction.

</details>

# Text Classification Revisited

We now build classifiers on top of word embeddings. Each classifier has two parts:

1. an **encoder**, which turns the text into one vector $\mathbf h(x)$,
2. a **softmax regressor** (an MLP), which turns $\mathbf h(x)$ into class probabilities.

We try two encoders: a **fixed** one (embed and pool), and a **trainable** one that reads words in order (RNN or LSTM).

## Tokenizer

The tokenizer should split text the same way the embeddings were trained. For GloVe and word2vec, a basic English tokenizer on lowercased text works well. A different embedding model may need a different tokenizer.

Our tokenizer has two jobs:

1. **Split** the text into tokens.
2. **Map** each token to its id in the embedding vocabulary (unknown words get the `<unk>` id).

What we keep depends on the encoder:

| encoder | punctuation and stopwords |
|---|---|
| embed and pool | removed: they add little to an average |
| RNN / LSTM | kept: word order and small words matter |

**Example.** *Dog bites man.* becomes `dog bites man` for embed and pool, and `dog bites man .` for an RNN.

## Batching and Padding

Models process several texts at once (a **batch**), which is much faster on a GPU. But texts have different lengths, so we **pad** the shorter ones with the `<pad>` id (here 0):

```
[1, 2, 3]       →  [1, 2, 3, 0]
[6, 2]          →  [6, 2, 0, 0]
[2, 1, 4, 5]    →  [2, 1, 4, 5]
```

Padding does two things:

- It makes the batch a rectangle (a 2D tensor of shape batch size × longest length).
- It marks which positions are real tokens: counting the non-zero ids gives the lengths $3, 2, 4$.

An encoder must ignore padding: a text should get the same vector alone or in a batch.

## Encoders

### Embed and Pool

The fixed encoder from before: look up each token's vector, then take the **mean** (or sum), skipping padding.

The vectors are now dense and pretrained, but it is still a bag of words: *dog bites man* and *man bites dog* get the same $\mathbf h(x)$.

### RNN Encoder

A **recurrent** encoder (RNN) reads the words one by one and keeps a **state** vector $\mathbf h$ that it updates at each word:

1. Start with $\mathbf h_0 = \mathbf 0$.
2. For each word $i$: $\mathbf h_i = \tanh(\mathbf W \mathbf e_i + \mathbf U \mathbf h_{i-1} + \mathbf b)$.
3. The last state $\mathbf h_l$ represents the whole text.

The parameters $\mathbf W$, $\mathbf U$ and $\mathbf b$ are the same at every step, so the RNN works for texts of any length, and its cost grows linearly with length. At padding positions, the state is simply kept.

**Example.** Take toy vectors *man* $= (1, 0, 0)$, *bites* $= (0, 1, 0)$, *dog* $= (0, 0, 1)$.

- Mean pooling gives $(\tfrac13, \tfrac13, \tfrac13)$ for both *man bites dog* and *dog bites man*.
- The RNN gives two different vectors, because each state depends on the words before it.

### LSTM Encoder

The simple RNN is hard to train: its gradients tend to vanish or explode. In practice we use an **LSTM** (or a GRU), which works the same way but is more stable.

- **Two vectors per step:** an LSTM keeps a hidden state and a cell.
- **Layers:** LSTMs can be stacked, one on top of another.
- **Word dropout:** normal dropout on the hidden states destroys the signal in long texts. Instead, we randomly zero out whole input word vectors.
- **Bidirectional:** one LSTM reads left to right, another right to left, and the two final states are joined.

## Softmax Regressor

The encoder's vector $\mathbf h(x)$ goes into an **MLP**, which outputs one score per class. A **log-softmax** turns the scores into log probabilities $\log p(k|x)$. We use log probabilities instead of probabilities because they are numerically more stable.

**Example.** An MLP with two hidden layers of size 50 and $K = 4$ classes:

$$\mathbf h(x) \to \text{Linear} \to \tanh \to \text{Linear} \to \tanh \to \text{Linear} \to \text{log-softmax} \to \log p(1|x), \ldots, \log p(4|x)$$

With embed and pool, *dog bites man* and *man bites dog* always get the same probabilities. With an LSTM, they can differ.

# Experiment

We compare the two classifiers on a real task: **embed and pool** (fixed encoder) and **LSTM** (trainable encoder), both using the same pretrained embeddings.

## Data

We use **AG News** again: news articles, each labelled with one of four topics: `world`, `sports`, `business`, `tech/sci-fi`. There are about 120,000 training articles and 7,600 test articles.

**Example.** *"Wall St. Bears Claw Back Into the Black. Short-sellers are seeing green again."* → `business`

**Splits:**

- **Training data:** split into $k = 5$ folds for **cross-validation**, so each design gets 5 accuracy scores instead of one.
- **Test data:** split into a small **devtest** set (100 articles per class, used for error analysis at the end) and the rest (kept unseen).

## Training

We train on one fold and check on its validation part after every epoch:

- **Validation loss going down** (and accuracy going up): the model is learning.
- **Validation loss going up** while training loss keeps going down: the model is **overfitting**. This is common with small data and flexible models like RNNs.

**Against overfitting:** dropout in linear layers, word dropout in recurrent layers, smaller hidden layers, weight decay (L2).

The LSTM is much slower than embed and pool, especially on a CPU. Use a GPU, or train on a subset of the data.

### When the LSTM Doesn't Win

The LSTM should usually beat embed and pool, since it sees word order. If it doesn't, check:

- **Data size:** apart from the embeddings, the LSTM is trained from scratch. A bigger model may need more data than we have.
- **Overfitting:** use more regularisation. If the embeddings are trained too, they are the largest part of the model and the most likely cause.
- **Hyperparameters:** hidden size, dropout, learning rate and so on are not learned by the optimiser. We have to choose them ourselves. (Bayesian optimisation and neural architecture search automate this, but are outside this course.)

### Cross-Validation

As in tutorial 1, we repeat training on every fold, and ideally with several random seeds.

Cross-validation is a **research tool**: it tells us which design choices are better. Once we have chosen, a final system can be trained on **all** the training data. If results vary a lot between seeds, we can train a few runs and combine them in an ensemble (as in tutorial 1).

### Comparing Designs

- **Use many folds and runs**, so differences are not just luck.
- **Structure the search:** we can't try every combination of settings. Vary one choice at a time, for example first the encoder type, then the hidden size.
- **Don't just average:** an average hides the spread. A box plot of all fold scores shows both the level and the uncertainty.

### Questions

**Question 1.** Do you think pretraining the embeddings made a difference for performance? Give reasons for and against.

<details>
<summary><b>Answer 1</b></summary>

**For:** pretrained vectors already know which words are similar. So a word that is rare in our training data still gets a useful vector, close to related words (*striker* near *football*).

**Against:** AG News is fairly large and topic words are frequent, so the model may learn enough from the training data alone. Only an experiment can tell.

</details>

**Question 2.** We could also start from a **random** embedding matrix and train it together with the classifier. Do you expect this to work? Give reasons for and against.

<details>
<summary><b>Answer 2</b></summary>

**For:** with enough data, the vectors become tuned to our task.

**Against:** the embedding matrix ($V \times D$) is by far the largest part of the model, so it overfits easily. Rare words get few updates, and words never seen in training keep their random vectors.

</details>

**Question 3.** In general, when do you expect pretraining to help most?

<details>
<summary><b>Answer 3</b></summary>

When there is **little labelled data**, when many words are **rare or unseen** in training, and for **bigger encoders** like LSTMs, which need more data. With lots of data, the gap shrinks. To test this, train on smaller and smaller subsets and compare pretrained with random embeddings. Without a pretrained model, we could also choose our own vocabulary or tokenizer, for example BPE as in tutorial 1.

</details>

## Analysis

Accuracy tells us **how often** a model is right, not **where** it fails. Error analysis asks which kinds of inputs cause errors.

**Method:**

1. Describe each devtest article with a few **features** that might relate to difficulty: length, rare words, readability, style, source, etc.
2. For each model, record whether its prediction was **correct** (1) or not (0).
3. Fit a **logistic regression** from the features to correctness. Each weight shows how much a feature relates to being right or wrong.

**Tools:** `sklearn`'s `LogisticRegression(solver='liblinear', penalty='l1')` pushes the weights of unhelpful features to 0. Below we use a small **Bayesian** version, which gives a whole distribution for each weight instead of a single number.

### Evaluating Trained Models

We load the saved models, predict the topics of the devtest articles, and print a **classification report**: precision, recall and F1 for each class. This shows which topics each model gets wrong most often.

### Features

As an example, we describe each article with:

- the **predicted class** (one-hot, so we can see if a model is better at some classes),
- four **readability** measures from `textstat`: word count, Flesch reading ease, reading time, and Flesch–Kincaid grade level.

**Example row:**

| system | label | pred | correct | words | reading ease | ... |
|---|---|---|---|---|---|---|
| lstm | business | business | 1 | 37 | 52.1 | ... |

Each article appears once per system, run and fold.

### The Bayesian Model

For observation $n$, we know:

- $c_n$: correct (1) or not (0),
- $s_n$: which system made the prediction,
- $\mathbf f_n$: the features of the input.

The model:

$$c_n \sim \operatorname{Bernoulli}(\sigma(\mu_n)), \qquad \mu_n = \phi + \delta_{s_n} + \mathbf w_{s_n}^\top \mathbf f_n$$

- $\phi$: the overall level of correctness,
- $\delta_s$: a **shift** for system $s$ (higher means a better system),
- $\mathbf w_s$: the **feature weights** for system $s$.

All parameters get a $\operatorname{Laplace}(0, 1)$ prior, which pushes them towards 0 unless the data says otherwise. The model works for any number of systems.

### Fitting with MCMC

The exact posterior can't be computed, so we draw approximate samples from it with **MCMC**. Before trusting the samples, check:

- **r_hat** close to 1.0 for every parameter (otherwise the chains didn't agree),
- **0 divergences** (or very few).

This model is simple, so problems are unlikely.

### Reading the Results

The posterior is shown as forest plots (with `arviz`):

- **System shifts** $\delta_s$: higher means the system is right more often. They are hard to read directly, since they act inside the sigmoid.
- **Accuracy:** easier to read, and close to what the box plots showed.
- **Feature weights** $\mathbf w_s$: the most interesting part. Features are standardised, so weights can be compared:
  - **close to 0:** the feature doesn't explain correctness,
  - **above 0:** the system does better as the feature grows,
  - **below 0:** the system struggles as the feature grows.

**Example.** A clearly negative weight for word count would mean the system makes more errors on long articles.

### Questions

**Question 1.** Choosing features is the scientist's job, not the statistics'. Which features would you test, and why? How would you compute them?

<details>
<summary><b>Answer 1</b></summary>

Some ideas:

- **Unknown-word rate:** the share of tokens that become `<unk>`. Unknown words carry no information, so this should hurt, maybe embed and pool most. Count `<unk>` ids after tokenizing.
- **Length:** short articles give little evidence. Count tokens.
- **Named entities:** names (*Microsoft*, *Iraq*) are strong topic hints. Count capitalised words, or use a named-entity tagger.
- **Topic overlap:** words typical of several topics (*market* in business and tech) may cause confusion. Count words that appear often in more than one class in the training data.

</details>

**Question 2.** Analyse the performance differences with your features. Does the data support your ideas? What would you look at next?

<details>
<summary><b>Answer 2</b></summary>

Fit the model with the new features, check r_hat and divergences, and look at the weights. For example, a clearly negative weight for unknown-word rate, larger for embed and pool than for the LSTM, would support the idea. Weights near 0 mean the idea isn't supported. Then look at examples of the errors to come up with new features, and repeat steps 1 and 2.

</details>